In [1]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 102.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 12.3 MB/s eta 0:00:00


In [2]:
import os
import io
import gc
import boto3
import json
import torch
import pandas as pd
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

In [4]:
# ==========================================
# 1. AWS CONFIGURATION (No hardcoding in prod!)
# ==========================================

AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY")

BUCKET_NAME = "meli-recsys-mvp-iturriago-844339186350-us-east-2-an"
REGION = "us-east-2"

# Initialize S3 Client. Think of this as your "file explorer" for the cloud.
s3_client = boto3.client('s3', region_name=REGION)

# ==========================================
# 2. PYTORCH & CLIP MODEL SETUP
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")

# We use CLIP because it projects both text and images into a shared latent space.
# This aligns perfectly with hybrid search requirements.
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

# Put model in evaluation mode
model.eval()

[INFO] Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [5]:
# ==========================================
# 3. HELPER FUNCTIONS (Memory Safe)
# ==========================================
def get_image_from_s3(s3_key: str):
    """Returns the PIL Image AND the byte buffer so they can be closed safely."""
    try:
        response = s3_client.get_object(Bucket=BUCKET_NAME, Key=s3_key)
        image_bytes = response['Body'].read()
        
        buffer = io.BytesIO(image_bytes)
        image = Image.open(buffer).convert("RGB")
        return image, buffer 
    except Exception as e:
        print(f"[WARNING] Could not load image {s3_key}: {e}")
        return None, None

def get_s3_image_keys(prefix="bronze/images/", max_keys=5000):
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=BUCKET_NAME, Prefix=prefix)
    
    image_keys = []
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                if obj['Key'].endswith(('.jpg', '.jpeg', '.png')):
                    image_keys.append(obj['Key'])
                if len(image_keys) >= max_keys:
                    return image_keys
    return image_keys

In [6]:
# ==========================================
# 4. EMBEDDING GENERATION (Production-Grade Stream-to-Disk)
# ==========================================
output_file = "image_embeddings.jsonl" 
s3_image_keys = get_s3_image_keys(max_keys=5000)

# Ensure we start with a clean slate
if os.path.exists(output_file):
    os.remove(output_file)

print(f"[INFO] Starting inference for {len(s3_image_keys)} items...")

# Open file in append mode to avoid keeping everything in RAM
with torch.no_grad(), open(output_file, 'a') as f:
    for i, s3_key in enumerate(s3_image_keys):
        
        # 1. Extract clean Article ID
        article_id = s3_key.split('/')[-1].rsplit('.', 1)[0]
        
        # 2. Get image and buffer from S3
        img, buffer = get_image_from_s3(s3_key)
        if img is None:
            continue
            
        try:
            # 3. Preprocessing
            inputs = processor(images=img, return_tensors="pt").to(device)
            
            # 4. Manual Forward Pass (The Senior Architect way to avoid shape bugs)
            # We go through the Vision Model to get the pooled output (768)
            vision_outputs = model.vision_model(pixel_values=inputs['pixel_values'])
            pooled_output = vision_outputs.pooler_output # Shape: [1, 768]
            
            # We project it to the multimodal space (512)
            image_features = model.visual_projection(pooled_output) # Shape: [1, 512]
            
            # 5. L2 Normalization (Essential for Cosine Similarity)
            image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
            
            # 6. Convert to flat list
            embedding_list = image_features.cpu().numpy()[0].tolist()
            
            # 7. Write to JSONL immediately
            payload = {
                "article_id": article_id,
                "image_embedding": embedding_list
            }
            f.write(json.dumps(payload) + '\n')
            
        except Exception as e:
            print(f"[ERROR] Critical failure at index {i} ({article_id}): {e}")
        
        finally:
            # 8. Aggressive Memory Cleanup
            img.close()
            buffer.close()
            # Explicitly delete large objects from local scope
            del inputs, vision_outputs, pooled_output, image_features, embedding_list, img, buffer
            
        # 9. Hardware synchronization and Garbage Collection
        if (i + 1) % 250 == 0:
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()
            print(f"   -> [PROGRESS] {i + 1}/{len(s3_image_keys)} processed. RAM is stable.")

[INFO] Starting inference for 5000 items...
   -> [PROGRESS] 250/5000 processed. RAM is stable.
   -> [PROGRESS] 500/5000 processed. RAM is stable.
   -> [PROGRESS] 750/5000 processed. RAM is stable.
   -> [PROGRESS] 1000/5000 processed. RAM is stable.
   -> [PROGRESS] 1250/5000 processed. RAM is stable.
   -> [PROGRESS] 1500/5000 processed. RAM is stable.
   -> [PROGRESS] 1750/5000 processed. RAM is stable.
   -> [PROGRESS] 2000/5000 processed. RAM is stable.
   -> [PROGRESS] 2250/5000 processed. RAM is stable.
   -> [PROGRESS] 2500/5000 processed. RAM is stable.
   -> [PROGRESS] 2750/5000 processed. RAM is stable.
   -> [PROGRESS] 3000/5000 processed. RAM is stable.
   -> [PROGRESS] 3250/5000 processed. RAM is stable.
   -> [PROGRESS] 3500/5000 processed. RAM is stable.
   -> [PROGRESS] 3750/5000 processed. RAM is stable.
   -> [PROGRESS] 4000/5000 processed. RAM is stable.
   -> [PROGRESS] 4250/5000 processed. RAM is stable.
   -> [PROGRESS] 4500/5000 processed. RAM is stable.
   ->

In [8]:
# ==========================================
# 5. UPLOAD FINAL SILVER LAYER TO S3
# ==========================================
print(f"\n[INFO] Uploading localized JSONL (Size: ~50MB) to S3...")
silver_s3_key = "silver/embeddings/image_embeddings.jsonl"
s3_client.upload_file(output_file, BUCKET_NAME, silver_s3_key)

print(f"[SUCCESS] Silver layer is now at s3://{BUCKET_NAME}/{silver_s3_key}")


[INFO] Uploading localized JSONL (Size: ~50MB) to S3...
[SUCCESS] Silver layer is now at s3://meli-recsys-mvp-iturriago-844339186350-us-east-2-an/silver/embeddings/image_embeddings.jsonl
